In [7]:
# Install the libraries
!pip install transformers datasets torch accelerate bitsandbytes -q

import torch
from datasets import load_dataset
import random
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

1. Loading Dataset
Total number of problems in the dataset: 1319


In [15]:
def main():
    #  Step 1: Dataset
    print("1. Loading Dataset")
    # Load the gsm8k dataset
    dataset = load_dataset("gsm8k", "main")


    # Select a random question from the test set for experiments
    problem_index = 350 # random
    problem_to_solve = dataset['test'][problem_index]
    print(f"Selected Problem (Index {problem_index}):")
    print(f"Question: {problem_to_solve['question']}")
    print(f"Correct Answer: {problem_to_solve['answer'].split('####')[-1].strip()}")
    print("-" * 20)

    #  Step 2: Model Selection
    print(" 2. Loading LLM ")
    # mistralai/Mistral-7B-Instruct-v0.2 is a good choice because of
    # its balance of performance and resource requirements.
    model_id = "mistralai/Mistral-7B-Instruct-v0.2"
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # Load model with quantisation to fit into smaller GPUs
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        load_in_8bit=True
    )

    # Create a text generation pipeline
    text_generator = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512, # setting this to 256 because outputs are getting truncated before completion in case of 128 tokens.
        do_sample=True,
        temperature=0.7,
        top_p=0.95
    )
    print(f"Model '{model_id}' loaded successfully.")
    print("-" * 20)

    #  Steps 3 & 4: Prompt Engineering and Refinement

    # Function to generate the response
    def generate_solution(prompt_template: str):
        messages = [{"role": "user", "content": prompt_template}]
        prompt = text_generator.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        outputs = text_generator(prompt)
        return outputs[0]["generated_text"].split("[/INST]")[-1].strip()

    #  Few-shot Examples
    # Sourced from the GSM8K training set for our prompts
    example_1_q = "Natalia is picking apples. She picks 3 apples on Monday and 2 more on Tuesday. On Wednesday, her dad gives her 4 apples. How many apples does she have now?"
    example_1_a_numeric = "9"
    example_1_a_cot = "Natalia starts with 3 apples. She picks 2 more, so she has 3 + 2 = 5 apples. Her dad gives her 4 more, so she now has 5 + 4 = 9 apples. The final answer is 9."

    example_2_q = "A bake sale has 200 cookies. They sell 150 cookies in the morning. Then, the bakers eat 10 of the remaining cookies. How many cookies are left?"
    example_2_a_numeric = "40"
    example_2_a_cot = "There are 200 cookies to start. They sell 150, so 200 - 150 = 50 cookies are left. The bakers eat 10, so 50 - 10 = 40 cookies are left. The final answer is 40."


    # Function for one shot prompting (numeric)
    def one_shot_prompting_numeric(problem: str) -> str:
        prompt = f"""
        Question: {example_1_q}
        Answer: {example_1_a_numeric}

        Question: {problem}
        Answer:
        """
        return generate_solution(prompt)

    # Function for two shot prompting (numeric)
    def two_shot_prompting_numeric(problem: str) -> str:
        prompt = f"""
        Question: {example_1_q}
        Answer: {example_1_a_numeric}

        Question: {example_2_q}
        Answer: {example_2_a_numeric}

        Question: {problem}
        Answer:
        """
        return generate_solution(prompt)

    # Function for two shot Chain of thoght (COT)
    def two_shot_cot_prompting(problem: str) -> str:
        prompt = f"""
        Question: {example_1_q}
        Answer: {example_1_a_cot}

        Question: {example_2_q}
        Answer: {example_2_a_cot}

        Question: {problem}
        Answer:
        """
        return generate_solution(prompt)

    # Function for refined COT prompt
    def refined_cot_prompting(problem: str) -> str:
        # Refinement - Add explicit instructions to think step by step and format the answer.
        prompt = f"""
        You are a math expert. Solve the following problems by thinking step-by-step.
        Explain your reasoning clearly and provide the final answer at the end.

        Question: {example_1_q}
        Answer: {example_1_a_cot}

        Question: {example_2_q}
        Answer: {example_2_a_cot}

        Question: {problem}
        Answer:
        """
        return generate_solution(prompt)


    #  Task 5: Evaluation
    print(" 3. Evaluating Prompting Techniques ")
    question_text = problem_to_solve['question']

    print("\n One Shot Prompting (Numeric) ")
    solution_one_shot = one_shot_prompting_numeric(question_text)
    print(f"Model Output:\n{solution_one_shot}")

    print("\n Two Shot Prompting (Numeric) ")
    solution_two_shot = two_shot_prompting_numeric(question_text)
    print(f"Model Output:\n{solution_two_shot}")

    print("\n Two Shot COT Prompting ")
    solution_cot = two_shot_cot_prompting(question_text)
    print(f"Model Output:\n{solution_cot}")

    print("\n Refined COT Prompting ")
    solution_refined = refined_cot_prompting(question_text)
    print(f"Model Output:\n{solution_refined}")
    print("\n Evaluation Complete ")

In [16]:
# Run the main function
if __name__ == "__main__":
    main()

1. Loading Dataset
Selected Problem (Index 350):
Question: Peyton scheduled after-work activities of a one hour yoga class on Monday, a cooking class that lasts three times as long as Monday’s yoga on Tuesday, a half-hour cheese-tasting event on Wednesday, a museum tour that takes half as long as the cooking class on Thursday, and two hours of errands on Friday. How many hours will all Peyton’s after-work activities take?
Correct Answer: 8
--------------------
 2. Loading LLM 


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Model 'mistralai/Mistral-7B-Instruct-v0.2' loaded successfully.
--------------------
 3. Evaluating Prompting Techniques 

 One Shot Prompting (Numeric) 


...

Model Output:
Let's calculate the hours for each activity:

Monday: 1 hour (yoga)
Tuesday: 3 hours (cooking class)
Wednesday: 1.5 hours (half hour for cheese-tasting and 1.5 hours for the rest of the day)
Thursday: T(cooking class) = 3 * 1 (Monday's yoga class length) = 3 hours (museum tour takes half the time) = 1.5 hours
Friday: 2 hours (errands)

Total hours: 1 + 3 + 1.5 + 3 + 1.5 + 2 = 11.5 hours

Since Peyton cannot have half an hour, we'll round up to the next full hour: 12 hours.

Answer: 12 hours.

 Two Shot Prompting (Numeric) 


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Model Output:
Let's calculate the hours for each activity:

Monday: 1 hour (yoga)
Tuesday: 3 hours (cooking class)
Wednesday: 1.5 hours (cheese-tasting)
Thursday: X hours (museum tour, half as long as cooking class)
Friday: 2 hours (errands)

We know the cooking class lasts 3 hours, so:

Thursday: X = 3 hours / 2 => X = 1.5 * 2 => X = 3 hours

Total hours: 1 hour (Monday) + 3 hours (Tuesday) + 1.5 hours (Wednesday) + 3 hours (Thursday) + 2 hours (Friday) => Total hours = 10 hours

Answer: 10 hours.

 Two Shot COT Prompting 


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Model Output:
On Monday, Peyton has a one-hour yoga class, so she spends 1 hour on that activity.
On Tuesday, she has a cooking class that lasts three times as long as Monday's yoga class, so she spends 1 hour (for yoga) x 3 = 3 hours on the cooking class.
On Wednesday, she attends a half-hour cheese-tasting event, so she spends 0.5 hours on that activity.
On Thursday, she goes on a museum tour that takes half as long as the cooking class, so she spends 3 hours (for cooking class) / 2 = 1.5 hours on the museum tour.
On Friday, she has 2 hours of errands to do.
Adding up all the hours, Peyton will spend 1 hour (yoga) + 3 hours (cooking class) + 0.5 hours (cheese-tasting) + 1.5 hours (museum tour) + 2 hours (errands) = 8 hours and 30 minutes in total for all her after-work activities. However, since the question asks for hours without minutes, we can round up to the nearest hour and say that Peyton's after-work activities will take 9 hours.

 Refined COT Prompting 
Model Output:
On Monda

## Summary

#### Most effective prompting strategies you discovered
#### Challenges you encountered
#### Insights gained from the experiment.

The answer slowly comes close to the correct answer at every step:
- One sht
- Two shot
- Two shot COT
- Refined COT


The experiment made me learn that prompt engineering is indeed an important strategy in Generative AI. Initially, one-shot and two-shot prompts aiming for a direct numerical answer were unreliable (12 and 10, correct ans is 8). The model often misunderstood the question or made silent calculation errors, providing a confident but incorrect response.

The most significant turning point in this experiment came with COT (Chain of Thought) prompting. Forcing the model to change its step by step reasoning dramatically improved accuracy (8.5 hours -> very near to 8). But this introduced a new challenge: truncated outputs. The nature of Chain of thought responses is verbose, because of which the responses frequently hit the default token limit (max_new_tokens), cutting the response off before completion. Adjusting the `max_new_token` limit was a non-obvious step.

Even with refined prompts, the model sometimes fumbled during the final calculation (though it responded with the correct answer in this case), proving that no technique is foolproof. The key learning was that effective prompting is an iterative loop of refining both the prompt's structure and the model's generation settings. Guiding the model's process (few shot / chain of thought) is a major factor that should be considered for more effectiveness than just asking for a final answer.